In [2]:
from google.colab import files
files.upload()

Saving generate_data.py to generate_data (1).py


{'generate_data (1).py': b'"""\ngenerate_data.py\nSynthetic data generator for: Digital Wallet KYC Funnel & Retention Analytics\n\nProduces 5 RAW (deliberately messy) CSV files in data/raw/:\n    users.csv, kyc_events.csv, transactions.csv, app_events.csv, date_dimension.csv\n\nWorks in Google Colab or locally. Fixed random seed => fully reproducible.\nNeeds only numpy + pandas (both pre-installed in Colab).\n"""\nimport os\nimport numpy as np\nimport pandas as pd\n\n# ----------------------------------------------------------------------\n# 1. CONFIG\n# ----------------------------------------------------------------------\nSEED = 42\nN_USERS = 20_000\nSIGNUP_START = pd.Timestamp("2025-01-01")\nSIGNUP_END = pd.Timestamp("2026-06-30")\nSNAPSHOT = pd.Timestamp("2026-09-30 23:59:59")   # "today" inside the simulated world\nSNAP = SNAPSHOT.to_datetime64()\nOUT_DIR = "data/raw"\n\nrng = np.random.default_rng(SEED)\n\n# ----------------------------------------------------------------------\

In [3]:
!python generate_data.py

users.csv              20,300 rows
kyc_events.csv         16,792 rows
transactions.csv      101,816 rows
app_events.csv        250,261 rows
date_dimension.csv        730 rows

Ground-truth funnel (before messing the data) - compare with your SQL in Week 4:
  registered                20,000
  kyc_started               15,735   (78.7% of previous stage)
  kyc_approved              10,173   (64.7% of previous stage)
  first_successful_txn       7,747   (76.2% of previous stage)
  second_txn_within_30d      6,001   (77.5% of previous stage)


In [4]:
!ls -lh data/raw

total 21M
-rw-r--r-- 1 root root  13M Oct 25 09:20 app_events.csv
-rw-r--r-- 1 root root  37K Oct 25 09:20 date_dimension.csv
-rw-r--r-- 1 root root 972K Oct 25 09:20 kyc_events.csv
-rw-r--r-- 1 root root 6.6M Oct 25 09:20 transactions.csv
-rw-r--r-- 1 root root 984K Oct 25 09:20 users.csv


In [5]:
!zip -r raw_data.zip data/raw
from google.colab import files
files.download("raw_data.zip")

  adding: data/raw/ (stored 0%)
  adding: data/raw/kyc_events.csv (deflated 79%)
  adding: data/raw/date_dimension.csv (deflated 88%)
  adding: data/raw/transactions.csv (deflated 78%)
  adding: data/raw/users.csv (deflated 82%)
  adding: data/raw/app_events.csv (deflated 81%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
import os
import pandas as pd
if not os.path.exists("data/raw/users.csv"):
    os.system("git clone https://github.com/ghantadevisri/digital-wallet-kyc-analytics.git")
    os.chdir("digital-wallet-kyc-analytics")

users = pd.read_csv("data/raw/users.csv")
print(users.shape)
users["city"].value_counts()

(20300, 6)


,count
city,
Delhi,2843
Mumbai,2805
Bengaluru,2541
Hyderabad,1645
Pune,1490
Chennai,1275
Jaipur,1216
Lucknow,1126
Indore,1077


In [7]:
city_map = {
    "mumbai": "Mumbai", "bombay": "Mumbai",
    "delhi": "Delhi", "new delhi": "Delhi",
    "bengaluru": "Bengaluru", "bangalore": "Bengaluru", "blr": "Bengaluru",
    "hyderabad": "Hyderabad", "hyd": "Hyderabad",
    "chennai": "Chennai", "madras": "Chennai",
    "pune": "Pune",
    "kolkata": "Kolkata", "calcutta": "Kolkata",
    "ahmedabad": "Ahmedabad", "ahmadabad": "Ahmedabad",
    "jaipur": "Jaipur", "jpr": "Jaipur",
    "lucknow": "Lucknow",
    "indore": "Indore",
}

# 1) remove extra spaces, 2) make lowercase, 3) look up in the table
cleaned = users["city"].str.strip().str.lower().map(city_map)

# Safety check: any real value that our table didn't know about?
print("Values not in the table:", users.loc[cleaned.isna() & users["city"].notna(), "city"].unique())

# Blank cities become "Unknown"
users["city"] = cleaned.fillna("Unknown")
users["city"].value_counts()

Values not in the table: []


,count
city,
Delhi,3177
Mumbai,3133
Bengaluru,2836
Hyderabad,1832
Pune,1658
Chennai,1438
Jaipur,1353
Lucknow,1243
Indore,1198


In [8]:
def clean_column(series, mapping):
    """Strip spaces, lowercase, translate with the table, and report anything unknown."""
    cleaned = series.str.strip().str.lower().map(mapping)
    missing = series[cleaned.isna() & series.notna()].unique()
    print("Values not in the table:", missing)
    return cleaned

device_map = {
    "android": "Android", "android os": "Android",
    "ios": "iOS", "iphone": "iOS",
    "web": "Web", "web browser": "Web",
}

channel_map = {
    "organic": "Organic",
    "paid social": "Paid Social", "paid_social": "Paid Social", "paid-social": "Paid Social",
    "google ads": "Google Ads", "googleads": "Google Ads",
    "referral": "Referral", "refer a friend": "Referral",
    "influencer": "Influencer", "influencer campaign": "Influencer",
    "offline agent": "Offline Agent", "offline_agent": "Offline Agent", "agent": "Offline Agent",
}

users["device_type"] = clean_column(users["device_type"], device_map)
users["acquisition_channel"] = clean_column(users["acquisition_channel"], channel_map)

# age_band has no spelling problems, only blanks
users["age_band"] = users["age_band"].fillna("Unknown")

for col in ["device_type", "acquisition_channel", "age_band"]:
    print(users[col].value_counts(), "\n")

Values not in the table: []
Values not in the table: []
device_type
Android    13497
iOS         5749
Web         1054
Name: count, dtype: int64 

acquisition_channel
Organic          5646
Paid Social      4506
Google Ads       3629
Referral         2861
Influencer       2007
Offline Agent    1651
Name: count, dtype: int64 

age_band
25-34      7564
18-24      5619
35-44      3774
45-54      2031
55+         918
Unknown     394
Name: count, dtype: int64 



In [9]:
print("Rows before:", len(users))
print("Duplicate user_ids:", users["user_id"].duplicated().sum())

# keep the first row for each user_id, drop the repeats
users = users.drop_duplicates(subset="user_id", keep="first").reset_index(drop=True)

print("Rows after:", len(users))
print("Unique user_ids:", users["user_id"].nunique())
print("Unknown cities:", (users["city"] == "Unknown").sum())
print("Unknown ages:", (users["age_band"] == "Unknown").sum())

# save the clean file
import os
os.makedirs("data/clean", exist_ok=True)
users.to_csv("data/clean/users_clean.csv", index=False)
print("Saved data/clean/users_clean.csv")

Rows before: 20300
Duplicate user_ids: 300
Rows after: 20000
Unique user_ids: 20000
Unknown cities: 196
Unknown ages: 390
Saved data/clean/users_clean.csv


In [10]:
from google.colab import files
files.download("data/clean/users_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
kyc = pd.read_csv("data/raw/kyc_events.csv")
print("Rows:", len(kyc))
print(kyc["kyc_status"].value_counts())

Rows: 16792
kyc_status
APPROVED       9438
REJECTED       3952
PENDING        2189
APPROVE         192
Verified        186
Approved        182
approved        175
Failed           87
REJECT           79
rejected         68
pending          68
Rejected         65
Pending          56
In Progress      55
Name: count, dtype: int64


In [12]:
status_map = {
    "approved": "APPROVED", "approve": "APPROVED", "verified": "APPROVED",
    "rejected": "REJECTED", "reject": "REJECTED", "failed": "REJECTED",
    "pending": "PENDING", "in progress": "PENDING",
}

kyc["kyc_status"] = clean_column(kyc["kyc_status"], status_map)
print(kyc["kyc_status"].value_counts())

# a rejection reason should exist only for REJECTED rows
print(kyc.groupby("kyc_status")["rejection_reason"].count())

Values not in the table: []
kyc_status
APPROVED    10173
REJECTED     4251
PENDING      2368
Name: count, dtype: int64
kyc_status
APPROVED       0
PENDING        0
REJECTED    4251
Name: rejection_reason, dtype: int64


In [13]:
# 1) turn the text dates into real dates so Python can compare them
kyc["kyc_started_at"] = pd.to_datetime(kyc["kyc_started_at"])
kyc["kyc_completed_at"] = pd.to_datetime(kyc["kyc_completed_at"])

signup = users[["user_id", "signup_date"]].copy()
signup["signup_date"] = pd.to_datetime(signup["signup_date"])

# 2) attach each user's signup date to their KYC rows
kyc = kyc.merge(signup, on="user_id", how="left")
print("KYC rows with no matching user:", kyc["signup_date"].isna().sum())

# 3) the two impossible cases
completed_before_signup = kyc["kyc_completed_at"].dt.normalize() < kyc["signup_date"]
completed_before_start = kyc["kyc_completed_at"] < kyc["kyc_started_at"]
print("Completed before signup:", completed_before_signup.sum())
print("Completed before it started:", completed_before_start.sum())

# 4) flag them (1 = problem, 0 = fine), then drop the helper column
kyc["has_date_issue"] = (completed_before_signup | completed_before_start).astype(int)
print("Flagged rows:", kyc["has_date_issue"].sum())
kyc = kyc.drop(columns="signup_date")

# 5) save
kyc.to_csv("data/clean/kyc_events_clean.csv", index=False)
print("Rows saved:", len(kyc))
kyc.head()

KYC rows with no matching user: 0
Completed before signup: 86
Completed before it started: 86
Flagged rows: 86
Rows saved: 16792


,user_id,kyc_started_at,kyc_completed_at,kyc_status,rejection_reason,has_date_issue
0,U000001,2025-01-01 01:16:16,2025-01-01 17:00:23,REJECTED,Blurry document,0
1,U000003,2025-01-01 07:29:39,2025-01-01 11:33:42,REJECTED,Address proof missing,0
2,U000007,2025-01-01 08:21:40,2025-01-03 00:05:13,REJECTED,Selfie mismatch,0
3,U000004,2025-01-01 08:22:20,NaT,PENDING,NaN,0
4,U000014,2025-01-01 14:15:31,NaT,PENDING,NaN,0


In [14]:
from google.colab import files
files.download("data/clean/kyc_events_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
tx = pd.read_csv("data/raw/transactions.csv")
print("Rows before:", len(tx))
print(tx["status"].value_counts(), "\n")
print(tx["payment_type"].value_counts())

Rows before: 101816
status
SUCCESS       91049
FAILED         5750
success        1266
SUCCESSFUL     1260
Success        1195
REVERSED        979
Failed          102
failed           95
FAIL             77
reversed         24
Reversed         19
Name: count, dtype: int64 

payment_type
UPI Transfer        34138
Merchant Payment    29137
Bill Payment        14668
Mobile Recharge     11700
Wallet Top-up        8027
Merchant Pay          639
merchant              623
UPI                   491
Upi transfer          480
upi                   469
Bill payment          324
bill pay              296
recharge              232
Mobile recharge       222
topup                 127
Wallet Topup          123
Top-up                120
Name: count, dtype: int64


In [16]:
status_map = {
    "success": "SUCCESS", "successful": "SUCCESS",
    "failed": "FAILED", "fail": "FAILED",
    "reversed": "REVERSED",
}

payment_map = {
    "upi transfer": "UPI Transfer", "upi": "UPI Transfer",
    "merchant payment": "Merchant Payment", "merchant": "Merchant Payment", "merchant pay": "Merchant Payment",
    "bill payment": "Bill Payment", "bill pay": "Bill Payment",
    "mobile recharge": "Mobile Recharge", "recharge": "Mobile Recharge",
    "wallet top-up": "Wallet Top-up", "wallet topup": "Wallet Top-up", "topup": "Wallet Top-up", "top-up": "Wallet Top-up",
}

tx["status"] = clean_column(tx["status"], status_map)
tx["payment_type"] = clean_column(tx["payment_type"], payment_map)

# remove payments that were recorded twice
print("Duplicate transaction_ids:", tx["transaction_id"].duplicated().sum())
tx = tx.drop_duplicates(subset="transaction_id", keep="first").reset_index(drop=True)
print("Rows after:", len(tx))

print(tx["status"].value_counts(), "\n")
print(tx["payment_type"].value_counts())

Values not in the table: []
Values not in the table: []
Duplicate transaction_ids: 405
Rows after: 101411
status
SUCCESS     94400
FAILED       5992
REVERSED     1019
Name: count, dtype: int64 

payment_type
UPI Transfer        35440
Merchant Payment    30289
Bill Payment        15225
Mobile Recharge     12104
Wallet Top-up        8353
Name: count, dtype: int64


In [17]:
# 1) test accounts: internal QA users, not real customers
is_test = tx["user_id"].str.startswith("TEST_")
print("Test-account rows:", is_test.sum())
tx = tx[~is_test].reset_index(drop=True)
print("Rows after removing test accounts:", len(tx))

# 2) every remaining payment should belong to a real user
known = tx["user_id"].isin(users["user_id"])
print("Payments with no matching user:", (~known).sum())

# 3) zero-value payments: flag, don't delete (1 = problem, 0 = fine)
tx["has_amount_issue"] = (tx["amount"] <= 0).astype(int)
print("Zero-value payments:", tx["has_amount_issue"].sum())
print(tx[tx["has_amount_issue"] == 1]["status"].value_counts())

print(tx["status"].value_counts())

# 4) save
tx.to_csv("data/clean/transactions_clean.csv", index=False)
print("Rows saved:", len(tx))

Test-account rows: 120
Rows after removing test accounts: 101291
Payments with no matching user: 0
Zero-value payments: 25
status
SUCCESS    25
Name: count, dtype: int64
status
SUCCESS     94280
FAILED       5992
REVERSED     1019
Name: count, dtype: int64
Rows saved: 101291


In [18]:
from google.colab import files
files.download("data/clean/transactions_clean.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
import os
import pandas as pd
from openpyxl.styles import Font

# raw ("before") tables
raw_users = pd.read_csv("data/raw/users.csv")
raw_kyc = pd.read_csv("data/raw/kyc_events.csv")
raw_tx = pd.read_csv("data/raw/transactions.csv")
raw_ev = pd.read_csv("data/raw/app_events.csv")

def nonstandard(raw_series, allowed):
    """rows with a value that is not written exactly in the standard form (blanks not counted)"""
    return int((raw_series.notna() & ~raw_series.isin(allowed)).sum())

issues = [
    # table, check, rows affected, action taken
    ("users", "Duplicate user_id rows", int(raw_users["user_id"].duplicated().sum()), "Removed (kept first row per user)"),
    ("users", "city spelled non-standard (e.g. Bombay, mumbai)", nonstandard(raw_users["city"], set(users["city"]) - {"Unknown"}), "Standardized to 11 cities"),
    ("users", "city blank", int(raw_users["city"].isna().sum()), "Filled with 'Unknown'"),
    ("users", "device_type spelled non-standard", nonstandard(raw_users["device_type"], set(users["device_type"])), "Standardized to Android / iOS / Web"),
    ("users", "acquisition_channel spelled non-standard", nonstandard(raw_users["acquisition_channel"], set(users["acquisition_channel"])), "Standardized to 6 channels"),
    ("users", "age_band blank", int(raw_users["age_band"].isna().sum()), "Filled with 'Unknown'"),
    ("kyc_events", "kyc_status spelled non-standard", nonstandard(raw_kyc["kyc_status"], {"APPROVED", "REJECTED", "PENDING"}), "Standardized to APPROVED / REJECTED / PENDING"),
    ("kyc_events", "Completed before signup or before start", int(kyc["has_date_issue"].sum()), "Kept, flagged in has_date_issue"),
    ("transactions", "Duplicate transaction_id rows", int(raw_tx["transaction_id"].duplicated().sum()), "Removed (kept first)"),
    ("transactions", "status spelled non-standard", nonstandard(raw_tx["status"], {"SUCCESS", "FAILED", "REVERSED"}), "Standardized to SUCCESS / FAILED / REVERSED"),
    ("transactions", "payment_type spelled non-standard", nonstandard(raw_tx["payment_type"], set(tx["payment_type"])), "Standardized to 5 payment types"),
    ("transactions", "Test-account rows (TEST_USER_xx)", int(raw_tx.drop_duplicates("transaction_id")["user_id"].str.startswith("TEST_").sum()), "Removed"),
    ("transactions", "Zero-value payments", int(tx["has_amount_issue"].sum()), "Kept, flagged in has_amount_issue"),
    ("app_events", "session_id blank", int(raw_ev["session_id"].isna().sum()), "Not cleaned yet - handled in SQL (Week 3)"),
    ("app_events", "Exact duplicate rows", int(raw_ev.duplicated().sum()), "Not cleaned yet - handled in SQL (Week 3)"),
]
summary = pd.DataFrame(issues, columns=["Table", "Check", "Rows affected", "Action taken"])

before_after = pd.DataFrame({
    "Table": ["users", "kyc_events", "transactions", "app_events"],
    "Rows before": [len(raw_users), len(raw_kyc), len(raw_tx), len(raw_ev)],
    "Rows after": [len(users), len(kyc), len(tx), len(raw_ev)],
    "Flag columns added": ["-", "has_date_issue", "has_amount_issue", "-"],
})
before_after["Rows removed"] = before_after["Rows before"] - before_after["Rows after"]
before_after = before_after[["Table", "Rows before", "Rows after", "Rows removed", "Flag columns added"]]

pairs = [("users", "city", raw_users["city"], users["city"]),
         ("users", "device_type", raw_users["device_type"], users["device_type"]),
         ("users", "acquisition_channel", raw_users["acquisition_channel"], users["acquisition_channel"]),
         ("kyc_events", "kyc_status", raw_kyc["kyc_status"], kyc["kyc_status"]),
         ("transactions", "status", raw_tx["status"], tx["status"]),
         ("transactions", "payment_type", raw_tx["payment_type"], tx["payment_type"])]
values = pd.DataFrame([(t, c, r.nunique(), a.nunique()) for t, c, r, a in pairs],                      columns=["Table", "Column", "Distinct values before", "Distinct values after"])

os.makedirs("excel", exist_ok=True)
with pd.ExcelWriter("excel/data_quality_report.xlsx", engine="openpyxl") as writer:
    for name, df in [("Issues_Found", summary), ("Before_After", before_after), ("Value_Standardization", values)]:
        df.to_excel(writer, sheet_name=name, index=False)
        ws = writer.sheets[name]
        ws.freeze_panes = "A2"
        for col in ws.columns:   # bold header, columns wide enough
            col[0].font = Font(bold=True)
            ws.column_dimensions[col[0].column_letter].width = max(len(str(c.value)) for c in col if c.value is not None) + 3

print(summary.to_string(index=False)); print(); print(before_after.to_string(index=False)); print(); print(values.to_string(index=False))

       Table                                           Check  Rows affected                                  Action taken
       users                          Duplicate user_id rows            300             Removed (kept first row per user)
       users city spelled non-standard (e.g. Bombay, mumbai)           2088                     Standardized to 11 cities
       users                                      city blank            200                         Filled with 'Unknown'
       users                device_type spelled non-standard           1647           Standardized to Android / iOS / Web
       users        acquisition_channel spelled non-standard           2061                    Standardized to 6 channels
       users                                  age_band blank            394                         Filled with 'Unknown'
  kyc_events                 kyc_status spelled non-standard           1213 Standardized to APPROVED / REJECTED / PENDING
  kyc_events         Com

In [21]:
from google.colab import files
files.download("excel/data_quality_report.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>